# FloodOps SG — RL Flood Prediction


**Pipeline:**
1. Install deps
2. Define the Gym environment (matches live sensor features in the app)
3. Train DQN → `flood_policy.pth`
4. Export → `flood_policy.onnx`
5. Copy `flood_policy.onnx` to `public/` in the repo

____
### 1. Imports

In [ ]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces
import onnxruntime as ort   # pip install onnxruntime if needed
import numpy as np
import torch
import torch.nn as nn
import random
import torch.optim as optim
from collections import deque


____
### 2. Gym Environment
- simulates a single Singapore flood zone over up to 60 timesteps
- the agent observes 6 live sensor features each step and must choose an alert level
- episode ends early if a flood fires and the agent issued no alert (action = 0)

**State (6 features):**
| Feature | Source in app | Normalisation |
|---------|--------------|---------------|
| `rainfallMm` | data.gov.sg `/rainfall` | ÷ 150 |
| `waterLevelPercent` | `zone.waterLevelPercent` | ÷ 100 |
| `trend_rising` | derived from rainfall delta | one-hot |
| `trend_stable` | derived from rainfall delta | one-hot |
| `trend_falling` | derived from rainfall delta | one-hot |
| `official` | signal type === 'official' | 0 / 1 |

**Actions:** 0 = no alert · 1 = watch · 2 = warning · 3 = flash flood alert

**Reward:** correct early escalation = +10, false alarm = −8, missed flood = −20

**Termination:** `step ≥ 60` OR `flood fired AND action == 0` (uncaught flood)

#### 2.1 FloodEnv Class

In [ ]:
class FloodEnv(gym.Env):
    metadata = {"render_modes": []}
    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Box(
            low=np.zeros(6, dtype=np.float32),
            high=np.ones(6, dtype=np.float32))
        self.action_space = spaces.Discrete(4)
        # 0 - No Alert, 1 - watch, 2 - warning, 3 - flash flood alert
        self.max_steps = 60

    def _obs(self):
        trend_vec = [0.0, 0.0, 0.0]
        trend_vec[["rising", "stable", "falling"].index(self._trend)] = 1.0
        return np.array([
            self._rain  / 150.0,
            self._water / 100.0,
            *trend_vec,
            float(self._official),
        ], dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._rain     = self.np_random.uniform(0, 80)
        self._water    = self.np_random.uniform(10, 60)
        self._trend    = self.np_random.choice(["rising", "stable", "falling"])
        self._official = self.np_random.random() < 0.1
        self._step     = 0
        self._prev_rain = self._rain
        return self._obs(), {}

    def step(self, action):
        self._step += 1
        delta = (self.np_random.uniform(2,  10) if self._trend == "rising"  else
                 self.np_random.uniform(-8,  0) if self._trend == "falling" else
                 self.np_random.uniform(-3,  3))
        self._rain  = float(np.clip(self._rain + delta, 0, 150))
        self._water = float(np.clip(
            self._water + (self.np_random.uniform(0, 3) if self._rain > 40
                           else self.np_random.uniform(-1, 1)), 0, 100))
        diff = self._rain - self._prev_rain
        self._trend = "rising" if diff > 3 else "falling" if diff < -3 else "stable"
        self._prev_rain = self._rain
        self._official = self._official or (
            self._rain > 80 and self.np_random.random() < 0.2)
        risk  = self._risk()
        flood = risk > 0.65 and self.np_random.random() < risk
        uncaught_flood = flood and action == 0  # flood fired with no alert issued
        done  = self._step >= self.max_steps or uncaught_flood
        if flood:
            reward = {3: 10.0, 2: 5.0, 1: 2.0, 0: -20.0}[action]
        else:
            if   action == 3 and risk < 0.3: reward = -8.0
            elif action == 2 and risk < 0.2: reward = -4.0
            elif action == 0 and risk < 0.3: reward =  0.5
            else:                             reward =  0.0
        return self._obs(), reward, done, False, {"flood": flood, "uncaught_flood": uncaught_flood}

    def _risk(self):
        return (0.4 * min(self._rain  / 100.0, 1.0)
              + 0.3 * min(self._water / 100.0, 1.0)
              + 0.1 * float(self._official))


In [ ]:
#sanity check
env = FloodEnv()
obs, _ = env.reset(seed=42)
print("obs shape:", obs.shape, "| sample obs:", obs.round(3))
obs2, r, done, _, info = env.step(0)
print("step reward:", r, "| done:", done, "| flood:", info["flood"])

##### 2.11 Explanation of code

```python
def __init__(self):
    self.observation_space = spaces.Box(low=0, high=1, shape=(6,))
    self.action_space = spaces.Discrete(4)
    self.max_steps = 60
```
- `spaces.Box(low=0, high=1, shape=(6,))` : declares the observation as a 6-element float array, all values normalised to [0, 1]
- `spaces.Discrete(4)` : declares 4 discrete actions (0 = no alert → 3 = flash flood)
- `self.max_steps = 60` : hard cap — episode terminates after 60 steps if no uncaught flood ends it sooner

```python
def _obs(self):
    trend_vec[["rising","stable","falling"].index(self._trend)] = 1.0
    return np.array([rain/150, water/100, *trend_vec, float(official)])
```
- converts internal floats into a normalised numpy array the network can consume
- trend is one-hot encoded so the network sees three separate binary signals

```python
def _risk(self):
    return 0.4 * min(rain/100, 1) + 0.3 * min(water/100, 1) + 0.1 * official
```
- scalar risk score in [0, 0.8] — mirrors `zoneRiskScore()` in the live TypeScript app
- flood is only possible when `risk > 0.65`, and fires with probability equal to `risk`

```python
uncaught_flood = flood and action == 0
done = self._step >= self.max_steps or uncaught_flood
```
- `uncaught_flood` : flood fired but the agent issued no alert (action == 0) — episode ends immediately with −20 reward
- a caught flood (action ≥ 1) does **not** end the episode — the agent is penalised but simulation continues
- this makes missing a flood a hard game-over, forcing the policy to always issue at least some alert

```python
return self._obs(), reward, done, False, {"flood": flood, "uncaught_flood": uncaught_flood}
```
- `info["flood"]` : True whenever a flood fired (caught or not) — used to compute detection rate in `evaluate`
- `info["uncaught_flood"]` : True only when the episode terminated due to a missed flood

##### 2.12 Over-Arching View:
```
reset()  →  randomise rain, water, trend, official  →  _obs()
                                                            │
step(action) ──► evolve dynamics ──► _risk() ──► flood? ──► reward ──► _obs()
                                                    │
                                    done=True if step≥60
                                    done=True if flood AND action==0  (uncaught)
                                    done=False if flood AND action≥1  (caught — continues)
```
- every episode starts with different initial conditions so the agent generalises across risk levels
- the asymmetric reward (+10 correct escalation / −20 missed flood) pushes the policy to escalate early
- the early-termination on uncaught floods amplifies the penalty — missing a flood also cuts the episode short, losing all future reward

____
### 3. DQN Model

#### 3.1 DQN Architecture
- learns a Q-function `Q(s, a)` estimating total future reward for each (state, action) pair
- picks the action with the highest Q-value at every step: `action = argmax_a Q(s, a)`
- trained by minimising the Bellman error: `loss = MSE(Q(s,a),  r + γ · max_a' Q_target(s', a'))`
- `γ = 0.95` — near-future rewards weighted almost equally to immediate reward

In [ ]:
class DQN(nn.Module):
    """6 inputs → 64 → 64 → 4 Q-values (one per alert level)."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(6, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, 4),
        )
    def forward(self, x):
        return self.net(x)



##### 3.11 Explanation of code

```python
self.net = nn.Sequential(
    nn.Linear(6, 128), nn.ReLU(),
    nn.Linear(128, 128), nn.ReLU(),
    nn.Linear(128, 4),
)
```
- `nn.Linear(6, 128)` : first hidden layer — maps 6 normalised sensor features to 128 neurons (768 weights + 128 biases = 896 params)
- `nn.ReLU()` : non-linear activation — without this, stacking linear layers collapses to a single linear transform regardless of depth
- `nn.Linear(128, 128)` : second hidden layer — learns combinations of the first layer's patterns (16,384 + 128 = 16,512 params)
- `nn.Linear(128, 4)` : output layer — one raw Q-value per alert level (512 + 4 = 516 params); no activation so Q-values are unbounded
- **total: 17,924 trainable parameters**

```python
def forward(self, x):
    return self.net(x)
```
- `x` shape: `(batch, 6)` → output shape: `(batch, 4)`
- during action selection `argmax` is taken over the 4 outputs; during training `.gather(1, actions)` picks the Q-value for the chosen action

____
### 4. Replay Buffer

#### 4.1 Creating ReplayBuffer
- stores up to 10,000 transitions `(obs, action, reward, next_obs, done)` in a circular buffer
- when full, the oldest entry is silently overwritten — no episode tracking needed
- `sample()` returns randomly drawn batches as tensors ready for network training

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity=10_000):
        self.capacity = capacity
        self.buffer   = []
        self.position = 0  # write-head for circular overwrite

    def add(self, obs, action, reward, next_obs, done):
        transition = (obs, action, reward, next_obs, done)
        if len(self.buffer) < self.capacity:
            self.buffer.append(transition)
        else:
            self.buffer[self.position] = transition
            self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        indices = np.random.choice(len(self.buffer), batch_size, replace=False)
        batch   = [self.buffer[i] for i in indices]
        obs, actions, rewards, next_obs, dones = zip(*batch)
        return (
            torch.tensor(np.array(obs),      dtype=torch.float32),
            torch.tensor(np.array(actions),  dtype=torch.long),
            torch.tensor(np.array(rewards),  dtype=torch.float32),
            torch.tensor(np.array(next_obs), dtype=torch.float32),
            torch.tensor(np.array(dones),    dtype=torch.float32),
        )
    def __len__(self):
        return len(self.buffer)


##### 4.11 Explanation of code

```python
def add(self, obs, action, reward, next_obs, done):
    if len(self.buffer) < self.capacity:
        self.buffer.append(transition)
    else:
        self.buffer[self.position] = transition
        self.position = (self.position + 1) % self.capacity
```
- `if len(self.buffer) < self.capacity` : while the buffer has room, simply grow it
- once full, `self.buffer[self.position] = transition` : overwrites the oldest entry at the write-head
- `self.position = (self.position + 1) % self.capacity` : advances the write-head by 1, wrapping around when it reaches the end

```python
def sample(self, batch_size):
    indices = np.random.choice(len(self.buffer), batch_size, replace=False)
    batch   = [self.buffer[i] for i in indices]
    obs, actions, rewards, next_obs, dones = zip(*batch)
    return (torch.tensor(...), ...)
```
- `np.random.choice(..., replace=False)` : samples without replacement so no transition appears twice in the same batch
- `zip(*batch)` : transposes list-of-tuples into tuple-of-lists for tensor conversion
- returns 5 tensors — `actions` is `torch.long` for use with `.gather()`; all others are `torch.float32`

____
### 5. Training the DQN Model

#### 5.1 Hyperparameters

In [ ]:
EPISODES   = 5_000 #number of episodes training will run for 
BATCH      = 64 # transitions sampled from buffer per update step
LR         = 1e-3 # adam optimiser learning rate
GAMMA      = 0.95 # discount factor, how much the agent values future rewards
EPS_START  = 1.0 # starting epsilon value
EPS_MIN    = 0.01 # minimum epsilon 
EPS_DECAY  = 0.995 #amount epsilon decays by per ep
TARGET_UPD = 500 #episodes between target networks synced
REPLAY_CAP = 10_000 #max transitions stored in replay buffer 

#### 5.2 Training Loop

In [ ]:
def train():
    env = FloodEnv()
    policy = DQN()
    target = DQN()
    target.load_state_dict(policy.state_dict())
    target.eval()
    optimizer = optim.Adam(policy.parameters(), lr=LR)
    replay = ReplayBuffer(capacity=REPLAY_CAP)
    epsilon = EPS_START
    rewards_window = deque(maxlen=200)
    for ep in range(1, EPISODES + 1):
        obs, _ = env.reset()
        total  = 0.0
        while True:
            if random.random() < epsilon:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    action = policy(torch.tensor(obs).unsqueeze(0)).argmax().item()
            next_obs, reward, done, _, _ = env.step(action)
            replay.add(obs, action, reward, next_obs, float(done))
            obs    = next_obs
            total += reward
            if len(replay) >= BATCH:
                s, a, r, ns, d = replay.sample(BATCH)
                q_pred = policy(s).gather(1, a.unsqueeze(1)).squeeze()
                with torch.no_grad():
                    q_target = r + GAMMA * target(ns).max(1).values * (1 - d)
                loss = nn.functional.mse_loss(q_pred, q_target)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            if done:
                break
        epsilon = max(EPS_MIN, epsilon * EPS_DECAY)
        rewards_window.append(total)

        if ep % TARGET_UPD == 0:
            target.load_state_dict(policy.state_dict())
        if ep % 500 == 0:
            avg = sum(rewards_window) / len(rewards_window)
            print(f"Episode {ep:>5}/{EPISODES}  avg_reward={avg:+.2f}  eps={epsilon:.3f}")

    print("\nTraining complete.")
    return policy


##### 5.21 Explanation of code

```python
replay.add(obs, action, reward, next_obs, float(done))
```
- stores each transition into the `ReplayBuffer` after every environment step

```python
if len(replay) >= BATCH:
    s, a, r, ns, d = replay.sample(BATCH)
```
- waits until the buffer has at least `BATCH=64` transitions before starting training
- `replay.sample()` returns 5 tensors already — no manual `torch.tensor(...)` conversion needed

```python
q_pred = policy(s).gather(1, a.unsqueeze(1)).squeeze()
```
- `policy(s)` : forward pass gives Q-values for all 4 actions, shape `(64, 4)`
- `.gather(1, a.unsqueeze(1))` : selects the Q-value for the action that was actually taken
- `.squeeze()` : collapses to shape `(64,)` to match `q_target`

```python
q_target = r + GAMMA * target(ns).max(1).values * (1 - d)
```
- `target(ns).max(1).values` : best Q-value from the next state according to the frozen target network
- `* (1 - d)` : zeroes out future value when the episode ended (`done=1`), so terminal states only count immediate reward
- `torch.no_grad()` : target values are fixed labels — no gradients needed

```python
if ep % TARGET_UPD == 0:
    target.load_state_dict(policy.state_dict())
```
- hard-copies the policy weights into the target network every 500 episodes
- prevents the Bellman target from chasing itself (instability) by keeping the target frozen between syncs

#### 5.3 Evaluating the Model
- runs `n_episodes` episodes with a fully greedy policy (no exploration, ε = 0)
- tracks reward, flood detection rate, false alarm rate, and action distribution across episodes

In [ ]:
def evaluate(policy, n_episodes=200, seed=0):
    LABELS = ["no_alert", "watch", "warning", "flash_flood"]
    env    = FloodEnv()
    policy.eval()

    total_rewards = []
    action_counts = [0, 0, 0, 0]
    floods_total  = 0
    floods_caught = 0  # flood occurred and agent issued action >= 1
    false_alarms  = 0  # action == 3 when risk < 0.3

    for ep in range(n_episodes):
        obs, _    = env.reset(seed=seed + ep)
        ep_reward = 0.0

        while True:
            with torch.no_grad():
                action = policy(torch.tensor(obs).unsqueeze(0)).argmax().item()

            obs, reward, done, _, info = env.step(action)
            ep_reward += reward
            action_counts[action] += 1

            if info["flood"]:
                floods_total += 1
                if action >= 1:
                    floods_caught += 1
            if action == 3 and env._risk() < 0.3:
                false_alarms += 1

            if done:
                break

        total_rewards.append(ep_reward)

    avg_reward     = sum(total_rewards) / n_episodes
    detection_rate = floods_caught / floods_total if floods_total > 0 else float("nan")
    total_steps    = sum(action_counts)

    print(f"Evaluation over {n_episodes} episodes")
    print(f"  Avg reward      : {avg_reward:+.2f}")
    print(f"  Flood events    : {floods_total}")
    print(f"  Detection rate  : {detection_rate:.1%}  ({floods_caught}/{floods_total} floods with action >= 1)")
    print(f"  False alarms    : {false_alarms}  (action=3 when risk < 0.3)")
    print(f"  Action distribution:")
    for label, count in zip(LABELS, action_counts):
        print(f"    {label:<15}  {count:>5}  ({count / total_steps:.1%})")

    return {
        "avg_reward":     avg_reward,
        "detection_rate": detection_rate,
        "false_alarms":   false_alarms,
        "action_counts":  action_counts,
        "rewards":        total_rewards,
    }



##### 5.31 Explanation of code

```python
policy.eval()
```
- switches off dropout/batchnorm training behaviour and disables gradient tracking — required before any inference pass

```python
with torch.no_grad():
    action = policy(torch.tensor(obs).unsqueeze(0)).argmax().item()
```
- `torch.no_grad()` : no gradients computed — evaluation is purely forward passes
- `.unsqueeze(0)` : adds a batch dimension so the input shape is `(1, 6)` as the network expects
- `.argmax().item()` : picks the action with the highest Q-value as a plain Python int

```python
if info["flood"]:
    floods_total += 1
    if action >= 1:
        floods_caught += 1
```
- a flood is "caught" if the agent issued at least a watch (action ≥ 1) at the moment the flood fired
- `floods_caught / floods_total` gives the **detection rate** — the primary safety metric

```python
if action == 3 and env._risk() < 0.3:
    false_alarms += 1
```
- counts steps where the agent issued a flash flood alert under genuinely low-risk conditions
- high false alarm counts indicate the policy is over-triggering

```python
return { "avg_reward": ..., "detection_rate": ..., ... }
```
- returns a dict so results can be compared across checkpoints or used for plotting

#### 5.4 Training and evaluating the model

In [ ]:
policy = train()

In [ ]:
results = evaluate(policy)

____
### 6. Storing the DQN as a Class

#### 6.1 DQNAgent Class
- bundles all state, networks, and logic into one object — mirrors the `TD3Agent` pattern
- `act` : ε-greedy action selection
- `update` : one Bellman step (Double DQN) + hard target-network sync every `target_update` gradient steps
- `train(render=False)` : full training loop, returns `self` so calls can be chained
- `evaluate(render=False)` : greedy evaluation reporting reward, detection rate and false alarms
- `play(render=False)` : runs one episode and returns a step-by-step result dict
- `save(render=False)` : saves policy weights to a `.pth` checkpoint
- `load(render=False)` : classmethod — creates a new agent and loads weights from a `.pth` file

All methods accept `render=False`; pass `render=True` to enable printed output.

In [ ]:
class DQNAgent:
    def __init__(self, env=None, gamma=0.95, eps_start=1.0, eps_min=0.01,
                 eps_decay=0.995, batch_size=64, buffer_capacity=10_000,
                 lr=1e-3, target_update=500):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.env = env if env is not None else FloodEnv()
        self.policy = DQN().to(self.device)
        self.target = DQN().to(self.device)
        self.target.load_state_dict(self.policy.state_dict())
        self.target.eval()
        for param in self.target.parameters():
            param.requires_grad = False
        self.optimizer     = optim.Adam(self.policy.parameters(), lr=lr)
        self.replay_buffer = ReplayBuffer(capacity=buffer_capacity)
        self.gamma         = gamma
        self.eps           = eps_start
        self.eps_min       = eps_min
        self.eps_decay     = eps_decay
        self.batch_size    = batch_size
        self.target_update = target_update
        self.total_updates = 0

    def act(self, obs, explore=True):
        if explore and random.random() < self.eps:
            return self.env.action_space.sample()
        obs_t = torch.tensor(obs, dtype=torch.float32, device=self.device).unsqueeze(0)
        self.policy.eval()
        with torch.no_grad():
            action = self.policy(obs_t).argmax().item()
        self.policy.train()
        return action

    def update(self):
        if len(self.replay_buffer) < self.batch_size:
            return
        s, a, r, ns, d = self.replay_buffer.sample(self.batch_size)
        s, a, r, ns, d = s.to(self.device), a.to(self.device), r.to(self.device), ns.to(self.device), d.to(self.device)
        q_pred = self.policy(s).gather(1, a.unsqueeze(1)).squeeze()
        with torch.no_grad():
            best_actions = self.policy(ns).argmax(1, keepdim=True)
            q_target = r + self.gamma * self.target(ns).gather(1, best_actions).squeeze() * (1 - d)
        loss = nn.functional.mse_loss(q_pred, q_target)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        self.total_updates += 1
        if self.total_updates % self.target_update == 0:
            self.target.load_state_dict(self.policy.state_dict())

    def train(self, episodes=5_000, log_every=500, render=False):
        rewards_window = deque(maxlen=200)
        for ep in range(1, episodes + 1):
            obs, _ = self.env.reset()
            total  = 0.0
            while True:
                action = self.act(obs, explore=True)
                next_obs, reward, done, _, _ = self.env.step(action)
                self.replay_buffer.add(obs, action, reward, next_obs, float(done))
                obs    = next_obs
                total += reward
                self.update()
                if done:
                    break
            self.eps = max(self.eps_min, self.eps * self.eps_decay)
            rewards_window.append(total)
            if render and ep % log_every == 0:
                avg = sum(rewards_window) / len(rewards_window)
                print(f"Episode {ep:>5}/{episodes}  avg_reward={avg:+.2f}  eps={self.eps:.3f}")
        print("\nTraining complete.")
        return self

    def evaluate(self, n_episodes=31, seed=0, render=False):
        LABELS = ["no_alert", "watch", "warning", "flash_flood"]
        env = FloodEnv()
        self.policy.eval()
        total_rewards = []
        action_counts = [0, 0, 0, 0]
        floods_total  = 0
        floods_caught = 0
        false_alarms  = 0
        for ep in range(n_episodes):
            obs, _    = env.reset(seed=seed + ep)
            ep_reward = 0.0
            while True:
                action = self.act(obs, explore=False)
                obs, reward, done, _, info = env.step(action)
                ep_reward += reward
                action_counts[action] += 1
                if info["flood"]:
                    floods_total += 1
                    if action >= 1:
                        floods_caught += 1
                if action == 3 and env._risk() < 0.3:
                    false_alarms += 1
                if done:
                    break
            total_rewards.append(ep_reward)
        self.policy.train()
        avg_reward     = sum(total_rewards) / n_episodes
        detection_rate = floods_caught / floods_total if floods_total > 0 else float("nan")
        total_steps    = sum(action_counts)
        if render:
            print(f"Evaluation over {n_episodes} episodes")
            print(f"  Avg reward      : {avg_reward:+.2f}")
            print(f"  Flood events    : {floods_total}")
            print(f"  Detection rate  : {detection_rate:.1%}  ({floods_caught}/{floods_total} floods with action >= 1)")
            print(f"  False alarms    : {false_alarms}  (action=3 when risk < 0.3)")
            print(f"  Action distribution:")
            for label, count in zip(LABELS, action_counts):
                print(f"  {label:<15}  {count:>5}  ({count / total_steps:.1%})")
        return {
            "avg_reward":     avg_reward,
            "detection_rate": detection_rate,
            "false_alarms":   false_alarms,
            "action_counts":  action_counts,
            "rewards":        total_rewards,
        }

    def play(self, seed=None, render=False): #simulates for 1 day 
        LABELS = ["no_alert", "watch", "warning", "flash_flood"]
        TRENDS = ["rising", "stable", "falling"]
        env = FloodEnv()
        obs, _ = env.reset(seed=seed)
        self.policy.eval()
        total   = 0.0
        step    = 0
        actions = []
        if render:
            print(f"{'Step':>4}  {'Rain':>6}  {'Water':>6}  {'Trend':<8}  {'Risk':>5}  {'Action':<15}  {'Reward':>7}  Note")
            print("-" * 72)
        while True:
            step  += 1
            action = self.act(obs, explore=False)
            obs, reward, done, _, info = env.step(action)
            total += reward
            actions.append(action)
            if render:
                trend_idx = int(np.argmax(obs[2:5]))
                note = "*** FLOOD ***" if info["flood"] else ""
                print(
                    f"{step:>4}  "
                    f"{env._rain:>6.1f}  "
                    f"{env._water:>6.1f}  "
                    f"{TRENDS[trend_idx]:<8}  "
                    f"{env._risk():>5.2f}  "
                    f"{LABELS[action]:<15}  "
                    f"{reward:>+7.1f}  "
                    f"{note}"
                )
            if done:
                break
        self.policy.train()
        if render:
            print("-" * 72)
            print(f"Episode ended  |  steps: {step}  |  total reward: {total:+.1f}")
        return {"total_reward": total, "steps": step, "actions": actions}

    def save(self, path="flood_policy.pth", render=False):
        torch.save(self.policy.state_dict(), path)
        if render:
            print(f"Saved {path}")
        return self

    @classmethod
    def load(cls, path="flood_policy.pth", render=False, **kwargs):
        agent = cls(eps_start=0.0, **kwargs)
        state_dict = torch.load(path, map_location=agent.device)
        agent.policy.load_state_dict(state_dict)
        agent.target.load_state_dict(state_dict)
        agent.policy.eval()
        if render:
            print(f"Loaded {path}")
        return agent

##### 6.11 Explanation of code

```python
self.policy = DQN().to(self.device)
self.target = DQN().to(self.device)
self.target.load_state_dict(self.policy.state_dict())
for param in self.target.parameters():
    param.requires_grad = False
```
- creates two identical networks; `target` is a frozen copy that provides stable Bellman labels
- `requires_grad = False` ensures the target never accumulates gradients — it is only ever updated via hard copy

```python
def act(self, obs, explore=True):
    if explore and random.random() < self.eps:
        return self.env.action_space.sample()
    action = self.policy(obs_t).argmax().item()
```
- `explore=True` during training (ε-greedy), `explore=False` during evaluation (fully greedy)
- `self.policy.eval()` / `.train()` toggles batchnorm/dropout behaviour around each inference call

```python
def update(self):
    best_actions = self.policy(ns).argmax(1, keepdim=True)
    q_target = r + self.gamma * self.target(ns).gather(1, best_actions).squeeze() * (1 - d)
    if self.total_updates % self.target_update == 0:
        self.target.load_state_dict(self.policy.state_dict())
```
- Double DQN: `self.policy` picks the best next action, `self.target` evaluates it — reduces Q-value overestimation vs vanilla DQN
- hard-syncs the target every `target_update` gradient steps (not per episode)

```python
def train(self, episodes=5_000, log_every=500, render=False):
    if render and ep % log_every == 0: print(...)
    if render: print("Training complete.")
    return self
```
- `render=False` by default — silent training for use inside loops or pipelines
- `return self` allows chaining: `agent = DQNAgent().train()`

```python
def evaluate(self, n_episodes=31, seed=0, render=False):
    self.policy.eval()
    ...
    self.policy.train()
    if render: print(metrics)
    return { "avg_reward": ..., "detection_rate": ..., ... }
```
- always returns the metrics dict regardless of `render`
- `self.policy.train()` restores training mode so `agent.train()` can be called after evaluate without side effects

```python
def play(self, seed=None, render=False):
```
- runs one greedy episode; `render=True` prints the step-by-step table
- returns `{"total_reward", "steps", "actions"}` — useful for scripted simulations

```python
def save(self, path="flood_policy.pth", render=False):
    torch.save(self.policy.state_dict(), path)
```
- saves only the policy weights (not optimizer or buffer) — sufficient for inference and fine-tuning

```python
@classmethod
def load(cls, path="flood_policy.pth", render=False, **kwargs):
    agent = cls(eps_start=0.0, **kwargs)
    agent.policy.load_state_dict(torch.load(path, map_location=agent.device))
    agent.target.load_state_dict(state_dict)
```
- `@classmethod` — called as `DQNAgent.load("flood_policy.pth")` without needing an existing instance
- `eps_start=0.0` makes the loaded agent act greedily immediately; set `agent.eps = 0.1` before `agent.train()` to resume training
- `**kwargs` forwards any hyperparameter overrides (e.g. `gamma=0.99`) to `__init__`

#### 6.2 Train and Evaluate

In [ ]:
agent = DQNAgent().train(episodes = 20_000)


In [ ]:
eval = agent.evaluate(render = True)

In [ ]:
agent.play()

#### 6.3 Save PyTorch Checkpoint

In [ ]:
agent.save()

#### 6.4 Load from checkpoint

In [ ]:
agent = DQNAgent.load("flood_policy.pth")
agent.play(seed=42)

### 7. Performance of model for an episode 

In [ ]:
def simulation(n = 1):
    final = []
    model = DQNAgent.load("flood_policy.pth")
    for i in range(0, n):
        results = model.play(seed = i)
        final.append(results)
    return final

def print_sim_res(res):
    n = 1
    for day in res:
        print(f"day {n}:")
        print(day)
        n+=1

In [ ]:
month = simulation(31)

In [ ]:
print_sim_res(month)

___
### 7. Exporting model weights to json format for use in FloodOpsSg


In [ ]:
import json as json
weights = {name: param.detach().numpy().tolist() for name, param in agent.policy.named_parameters()}
with open("flood_policy_weights.json", "w") as f:
    json.dump(weights, f)

print("Keys:", list(weights.keys()))
print("Done → flood_policy_weights.json")